In [ ]:
import os
import glob
import pandas as pd
import numpy as np

# ----------------------------
# Configurazione
# ----------------------------
base_dir = "/home/damn/Documents/PROJECTS/GITHUB/HACKATON/Data4Econ_Hackaton/in-sample"
output_filename = "augmented_data_finale.csv"
output_path = os.path.join(base_dir, output_filename)


In [ ]:
def list_impianti_forecast(base_dir):
    """
    Ritorna la lista dei nomi degli impianti estraendo dai file
    '2024_meteo_fcs_<impianto>.csv'.
    """
    pattern = os.path.join(base_dir, "2024_meteo_fcs_*.csv")
    paths = glob.glob(pattern)
    impianti = []
    for path in paths:
        fname = os.path.basename(path)
        nome = fname.replace("2024_meteo_fcs_", "").replace(".csv", "")
        impianti.append(nome)
    return sorted(impianti)

impianti = list_impianti_forecast(base_dir)

In [ ]:
def load_and_merge_forecast(base_dir, impianti):
    """
    Per ogni impianto in 'impianti', carica '2024_meteo_fcs_<impianto>.csv',
    converte 'Dataora' in datetime, rinomina le colonne (tranne 'Dataora')
    aggiungendo suffisso _<impianto>, e fa merge inner su 'Dataora' per creare
    un unico DataFrame unito.
    """
    df_merged = None
    for imp in impianti:
        path = os.path.join(base_dir, f"2024_meteo_fcs_{imp}.csv")
        df_imp = pd.read_csv(path, sep=";")
        df_imp["Dataora"] = pd.to_datetime(df_imp["Dataora"])
        
        rename_map = {
            col: f"{col.lower()}_{imp}"
            for col in df_imp.columns if col != "Dataora"
        }
        df_imp = df_imp.rename(columns=rename_map)
        
        if df_merged is None:
            df_merged = df_imp.copy()
        else:
            df_merged = df_merged.merge(df_imp, on="Dataora", how="inner")
    return df_merged

df = load_and_merge_forecast(base_dir, impianti)

In [ ]:
def add_wds_wdd_features(df, impianti):
    """
    Per ogni impianto, dato che esistono colonne:
      - wds_<impianto>
      - wdd_<impianto>   (in gradi)
    Aggiunge per ciascun impianto le colonne:
      - wds2_<impianto> = wds^2
      - wds3_<impianto> = wds^3
      - u_<impianto>    = wds * cos(wdd * π/180)
      - v_<impianto>    = wds * sin(wdd * π/180)
    """
    for imp in impianti:
        wds_col = f"wds_{imp}"
        wdd_col = f"wdd_{imp}"
        if wds_col in df.columns and wdd_col in df.columns:
            df[f"wds2_{imp}"] = df[wds_col] ** 2
            df[f"wds3_{imp}"] = df[wds_col] ** 3
            radians = np.deg2rad(df[wdd_col])
            df[f"u_{imp}"] = df[wds_col] * np.cos(radians)
            df[f"v_{imp}"] = df[wds_col] * np.sin(radians)
        else:
            df[f"wds2_{imp}"] = np.nan
            df[f"wds3_{imp}"] = np.nan
            df[f"u_{imp}"]    = np.nan
            df[f"v_{imp}"]    = np.nan

add_wds_wdd_features(df, impianti)

In [ ]:
def add_time_cyclic_features(df):
    """
    Aggiunge le colonne:
      - sin_ora, cos_ora     → sin/cos(2π * hour / 24)
      - sin_giorno, cos_giorno → sin/cos(2π * day  / 31)
      - sin_mese, cos_mese   → sin/cos(2π * month/ 12)
    """
    df["Dataora"] = pd.to_datetime(df["Dataora"])
    ore    = df["Dataora"].dt.hour
    giorni = df["Dataora"].dt.day
    mesi   = df["Dataora"].dt.month

    df["sin_ora"]    = np.sin(2 * np.pi * ore    / 24)
    df["cos_ora"]    = np.cos(2 * np.pi * ore    / 24)
    df["sin_giorno"] = np.sin(2 * np.pi * giorni / 31)
    df["cos_giorno"] = np.cos(2 * np.pi * giorni / 31)
    df["sin_mese"]   = np.sin(2 * np.pi * mesi   / 12)
    df["cos_mese"]   = np.cos(2 * np.pi * mesi   / 12)

add_time_cyclic_features(df)

In [ ]:
def add_rolling_features(df, impianti, windows=[3, 6]):
    """
    Per ciascun impianto e per var in [wds, wds2, wds3], aggiunge rolling mean
    con finestra oraria di 3h e 6h:
      - <var>_<impianto>_rol3
      - <var>_<impianto>_rol6
    """
    df = df.sort_values("Dataora").reset_index(drop=True)
    for imp in impianti:
        for var in ["wds", "wds2", "wds3"]:
            col = f"{var}_{imp}"
            if col in df.columns:
                for w in windows:
                    df[f"{col}_rol{w}"] = df[col].rolling(window=w, min_periods=1).mean()
            else:
                for w in windows:
                    df[f"{col}_rol{w}"] = np.nan
    return df

df = add_rolling_features(df, impianti, windows=[3, 6])

In [ ]:
def fill_nan_rolling_with_original(df, impianti, vars_list=["wds", "wds2", "wds3"], windows=["rol3", "rol6"]):
    """
    Per ciascun impianto e per ogni var in vars_list, sostituisce i NaN
    nelle colonne <var>_<impianto>_<window> con i corrispondenti valori di <var>_<impianto>.
    """
    for imp in impianti:
        for var in vars_list:
            base_col = f"{var}_{imp}"
            if base_col not in df.columns:
                continue  # nessuna colonna originale, salto
            for window in windows:
                rolling_col = f"{base_col}_{window}"
                if rolling_col in df.columns:
                    df[rolling_col] = df[rolling_col].fillna(df[base_col])

fill_nan_rolling_with_original(df, impianti, vars_list=["wds", "wds2", "wds3"], windows=["rol3", "rol6"])


In [ ]:
def add_average_rolling_across_impianti(df, impianti, windows=[3, 6]):
    """
    Per ciascun var in [wds, wds2, wds3] e per ciascuna finestra (3, 6),
    calcola la media row-wise di tutte le colonne "<var>_<impianto>_rol<w>"
    e aggiunge le colonne:
      - media_rol3_wds_tutti, media_rol6_wds_tutti
      - media_rol3_wds2_tutti, media_rol6_wds2_tutti
      - media_rol3_wds3_tutti, media_rol6_wds3_tutti
    """
    for var in ["wds", "wds2", "wds3"]:
        for w in windows:
            pattern = rf"^{var}_.+_rol{w}$"
            cols = df.filter(regex=pattern).columns
            nome_media = f"media_rol{w}_{var}_tutti"
            if len(cols) > 0:
                df[nome_media] = df[cols].mean(axis=1)
            else:
                df[nome_media] = np.nan

add_average_rolling_across_impianti(df, impianti, windows=[3, 6])


In [ ]:
def add_rowwise_stats(df, prefissi):
    """
    Per ciascun prefisso (es. "wds_", "wds2_", "wds3_", "u_", "v_", "rnf_", "rhm_"),
    trova tutte le colonne che iniziano con quel prefisso e calcola media e std
    riga-per-riga, aggiungendo le colonne:
      - media_<prefisso>tutti
      - std_<prefisso>tutti
    """
    for pref in prefissi:
        cols = df.filter(regex=rf"^{pref}").columns
        nome_media = f"media_{pref}tutti".replace("__", "_")
        nome_std   = f"std_{pref}tutti".replace("__", "_")
        if len(cols) > 0:
            df[nome_media] = df[cols].mean(axis=1)
            df[nome_std]   = df[cols].std(axis=1)
        else:
            df[nome_media] = np.nan
            df[nome_std]   = np.nan

prefissi = ["wds_", "wds2_", "wds3_", "u_", "v_", "rnf_", "rhm_"]
add_rowwise_stats(df, prefissi)



In [ ]:
df.to_csv(output_path, index=False)
print(f"Dataset finale salvato in: {output_path}")